<a href="https://colab.research.google.com/github/GKSJ-AI-CliniScan/MedAssistAI/blob/TahuraShaikh/XG_Boost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import time
import warnings

warnings.filterwarnings("ignore")

# Load dataset
dataset = pd.read_csv("/content/dataset2_training.csv")

print("Dataset Shape:", dataset.shape)
print("Number of Features:", dataset.shape[1] - 1)
print("Number of Diseases:", dataset["Disease"].nunique())

Dataset Shape: (60000, 378)
Number of Features: 377
Number of Diseases: 658


In [ ]:
X = dataset.drop(columns=["Disease"])
y = dataset["Disease"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)
print("Unique Diseases:", y.nunique())

X Shape: (60000, 377)
y Shape: (60000,)
Unique Diseases: 658


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)

print("Number of Classes:", len(label_encoder.classes_))
print("Minimum Label:", y_encoded.min())
print("Maximum Label:", y_encoded.max())

Number of Classes: 658
Minimum Label: 0
Maximum Label: 657


In [ ]:
from sklearn.model_selection import train_test_split

X_train_xgb, X_test_xgb, y_train_xgb, y_test_xgb = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("X_train:", X_train_xgb.shape)
print("X_test :", X_test_xgb.shape)

print("Train classes:", len(np.unique(y_train_xgb)))
print("Test classes :", len(np.unique(y_test_xgb)))

X_train: (48000, 377)
X_test : (12000, 377)
Train classes: 658
Test classes : 614


In [ ]:
print("Minimum train label:", y_train_xgb.min())
print("Maximum train label:", y_train_xgb.max())

print("Minimum test label:", y_test_xgb.min())
print("Maximum test label:", y_test_xgb.max())

print("Missing classes in training:",
      set(range(658)) - set(np.unique(y_train_xgb)))

print("Missing classes in testing:",
      len(set(range(658)) - set(np.unique(y_test_xgb))))

Minimum train label: 0
Maximum train label: 657
Minimum test label: 0
Maximum test label: 657
Missing classes in training: set()
Missing classes in testing: 44


In [ ]:
!pip install -q -U xgboost

In [ ]:
import xgboost as xgb

print("XGBoost version:", xgb.__version__)

XGBoost version: 3.3.0


In [ ]:
from xgboost import XGBClassifier

start = time.time()

xgb_model = XGBClassifier(
    objective="multi:softprob",
    num_class=658,
    n_estimators=50,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    eval_metric="mlogloss"
)

xgb_model.fit(
    X_train_xgb,
    y_train_xgb,
    verbose=False
)

end = time.time()

print("XGBoost Training Time:",
      round(end - start, 2),
      "seconds")

XGBoost Training Time: 785.07 seconds


In [ ]:
start = time.time()

xgb_pred = xgb_model.predict(X_test_xgb)

end = time.time()

print("XGBoost Prediction Time:",
      round(end - start, 2),
      "seconds")

print("Unique Predictions:",
      len(np.unique(xgb_pred)))

print("Total Classes:",
      len(np.unique(y_encoded)))

XGBoost Prediction Time: 6.07 seconds
Unique Predictions: 355
Total Classes: 658


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

xgb_accuracy = accuracy_score(
    y_test_xgb,
    xgb_pred
)

xgb_precision = precision_score(
    y_test_xgb,
    xgb_pred,
    average="weighted",
    zero_division=0
)

xgb_recall = recall_score(
    y_test_xgb,
    xgb_pred,
    average="weighted",
    zero_division=0
)

xgb_f1 = f1_score(
    y_test_xgb,
    xgb_pred,
    average="weighted",
    zero_division=0
)

print("XGBoost Results")
print("----------------")
print("Accuracy :", round(xgb_accuracy, 4))
print("Precision:", round(xgb_precision, 4))
print("Recall   :", round(xgb_recall, 4))
print("F1 Score :", round(xgb_f1, 4))

XGBoost Results
----------------
Accuracy : 0.7677
Precision: 0.7332
Recall   : 0.7677
F1 Score : 0.7453


In [ ]:
xgb_results = {
    "Model": "XGBoost",
    "Accuracy": xgb_accuracy,
    "Precision": xgb_precision,
    "Recall": xgb_recall,
    "F1 Score": xgb_f1,
    "Training Time": round(end - start, 2)
}

print(xgb_results)

{'Model': 'XGBoost', 'Accuracy': 0.7676666666666667, 'Precision': 0.7332140284601816, 'Recall': 0.7676666666666667, 'F1 Score': 0.7453278518234948, 'Training Time': 6.07}


In [ ]:
import joblib

joblib.dump(
    xgb_model,
    "/content/xgboost_model.pkl"
)

print("XGBoost model saved successfully.")

XGBoost model saved successfully.
